## 대본에 있는 수치 재계산 (leak-free 모델 기준)

- 대본의 4번 섹션(심야 비율, 금액-사기 상관): 

amt, trans_hour, category, is_fraud는 누수 변수에 포함되지 않았어서 -> 두 수치(약 85%/98%, r=0.219 / 6개 업종 반전)는 정상 -> 그대로 유지, 혹시 모르니 재확인하긴 할 것.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# --- train_11features.csv (leak-free 재생성본) ---
CANDIDATE_TRAIN11_PATHS = [
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\train_11features.csv"),
    Path(r"C:\Users\user\Desktop\bdai부캠\BDAI\data\train_11features.csv"),
    Path("train_11features.csv"),
    Path("data/train_11features.csv"),
]

# --- test_11features.csv (leak-free 재생성본, 참고용) ---
CANDIDATE_TEST11_PATHS = [
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\test_11features.csv"),
    Path(r"C:\Users\user\Desktop\bdai부캠\BDAI\data\test_11features.csv"),
    Path("test_11features.csv"),
    Path("data/test_11features.csv"),
]

# --- tableau_export_leakfree.csv (08_rebuild_tableau_leakfree.py 결과물) ---
CANDIDATE_TABLEAU_PATHS = [
    Path(r"C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\tableau_export_leakfree.csv"),
    Path(r"C:\Users\user\Desktop\bdai부캠\BDAI\tableau\tableau_export_leakfree.csv"),
    Path("tableau_export_leakfree.csv"),
    Path("tableau/tableau_export_leakfree.csv"),
]


def find_path(candidates, label):
    for p in candidates:
        if p.exists():
            print(f"[{label}] 찾음: {p.resolve()}")
            return p
    raise FileNotFoundError(
        f"[{label}] 파일을 찾지 못했습니다. 아래 CANDIDATE_*_PATHS 에 실제 경로를 추가해주세요.\n"
        + "\n".join(str(p) for p in candidates)
    )


train_path = find_path(CANDIDATE_TRAIN11_PATHS, "train_11features.csv")
tableau_path = find_path(CANDIDATE_TABLEAU_PATHS, "tableau_export_leakfree.csv")

train_df = pd.read_csv(train_path, parse_dates=["trans_date_trans_time"])
tableau_df = pd.read_csv(tableau_path, parse_dates=["trans_date_trans_time"])

print(f"\ntrain_11features: {train_df.shape}")
print(f"tableau_export_leakfree: {tableau_df.shape}")
print("\ntableau_export_leakfree 컬럼:", list(tableau_df.columns))


[train_11features.csv] 찾음: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\train_11features.csv
[tableau_export_leakfree.csv] 찾음: C:\Users\splen\OneDrive\Desktop\BDAI_\BOOSTMAP\Fraud-FDS-Project\revision\tableau_export_leakfree.csv

train_11features: (1296675, 15)
tableau_export_leakfree: (1852394, 10)

tableau_export_leakfree 컬럼: ['trans_date_trans_time', 'category', 'merchant', 'amt', 'trans_hour', 'is_fraud', 'model_prob', 'tier_5_final', 'cluster_type', 'response_action']


In [3]:
## 4번 섹션 재계산 (1) — 심야 이상거래 비율 / 심야 거래 중 정상거래 비율

NIGHT_HOURS = set(range(22, 24)) | set(range(0, 4))  # 22, 23, 0, 1, 2, 3시

night_mask = train_df["trans_hour"].isin(NIGHT_HOURS)
fraud_mask = train_df["is_fraud"] == 1

TP = int((night_mask & fraud_mask).sum())   # 심야 · 실제사기
FP = int((night_mask & ~fraud_mask).sum())  # 심야 · 실제정상
FN = int((~night_mask & fraud_mask).sum())  # 주간 · 실제사기
TN = int((~night_mask & ~fraud_mask).sum()) # 주간 · 실제정상

pct_fraud_at_night = TP / (TP + FN) * 100
pct_night_is_normal = FP / (TP + FP) * 100

print(f"TP(심야·사기)={TP:,}  FP(심야·정상)={FP:,}  FN(주간·사기)={FN:,}  TN(주간·정상)={TN:,}")
print(f"\n전체 이상거래 중 심야 비율      : {pct_fraud_at_night:.2f}%  (원래 보고값: 약 85%)")
print(f"심야 거래 중 정상거래 비율       : {pct_night_is_normal:.2f}%  (원래 보고값: 약 98%)")

print("\n[대본용 문장]")
print(
    f"전체 이상거래의 약 {pct_fraud_at_night:.0f}%가 심야에 발생했지만, "
    f"심야 거래 자체의 약 {pct_night_is_normal:.0f}%는 정상거래였습니다."
)


TP(심야·사기)=6,362  FP(심야·정상)=298,520  FN(주간·사기)=1,144  TN(주간·정상)=990,649

전체 이상거래 중 심야 비율      : 84.76%  (원래 보고값: 약 85%)
심야 거래 중 정상거래 비율       : 97.91%  (원래 보고값: 약 98%)

[대본용 문장]
전체 이상거래의 약 85%가 심야에 발생했지만, 심야 거래 자체의 약 98%는 정상거래였습니다.


In [ ]:
## 4번 섹션 재계산 (2) — 금액×사기 상관 + 업종별 심슨의 역설
 
overall_r = train_df["amt"].corr(train_df["is_fraud"])

cat_corrs = (
    train_df.groupby("category")
    .apply(lambda g: g["amt"].corr(g["is_fraud"]))
    .rename("pearson_r")
    .sort_values()
)

reversed_categories = cat_corrs[cat_corrs < 0]

print(f"전체 상관계수 (amt vs is_fraud): r = {overall_r:.3f}  (원래 보고값: r=0.219)")
print(f"\n업종별 상관계수 (오름차순):")
print(cat_corrs)
print(f"\n방향이 반전된(음수) 업종 수: {len(reversed_categories)}개  (원래 보고값: 6개)")
print(reversed_categories)

print("\n[대본용 문장]")
print(
    f"전체 데이터에서는 거래금액과 이상거래 여부가 약한 양의 상관관계(r={overall_r:.3f})를 보였지만, "
    f"업종별로 나누자 {len(reversed_categories)}개 업종에서는 관계의 방향이 반대로 나타났습니다."
)

전체 상관계수 (amt vs is_fraud): r = 0.219  (원래 보고값: r=0.219)

업종별 상관계수 (오름차순):
category
gas_transport    -0.2211
grocery_net      -0.1001
kids_pets        -0.0372
health_fitness   -0.0275
personal_care    -0.0218
travel           -0.0089
food_dining       0.0574
misc_pos          0.0647
home              0.1642
shopping_pos      0.2938
entertainment     0.3309
grocery_pos       0.4383
shopping_net      0.4926
misc_net          0.5198
Name: pearson_r, dtype: float64

방향이 반전된(음수) 업종 수: 6개  (원래 보고값: 6개)
category
gas_transport    -0.2211
grocery_net      -0.1001
kids_pets        -0.0372
health_fitness   -0.0275
personal_care    -0.0218
travel           -0.0089
Name: pearson_r, dtype: float64

[대본용 문장]
전체 데이터에서는 거래금액과 이상거래 여부가 약한 양의 상관관계(r=0.219)를 보였지만, 업종별로 나누자 6개 업종에서는 관계의 방향이 반대로 나타났습니다.


C:\Users\splen\AppData\Local\Temp\ipykernel_8356\2773257661.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["amt"].corr(g["is_fraud"]))


In [5]:
## 7-1. FDS 위험대응 시뮬레이터 — 전체 탐지율 / 탐지 사기 건수 / 오탐 건수
## "Recall 96.82%｜탐지 사기 9,344건｜오탐 3,198건"

FLAG_TIERS = ["위험", "긴급"]

flagged = tableau_df["tier_5_final"].isin(FLAG_TIERS)
is_fraud = tableau_df["is_fraud"] == 1

total_fraud = int(is_fraud.sum())
detected_fraud = int((flagged & is_fraud).sum())
false_positive = int((flagged & ~is_fraud).sum())
false_positive_amt = tableau_df.loc[flagged & ~is_fraud, "amt"].sum()

recall = detected_fraud / total_fraud * 100

print(f"전체 실제 사기 건수     : {total_fraud:,}")
print(f"탐지된 사기 건수(TP)    : {detected_fraud:,}")
print(f"오탐 건수(FP)           : {false_positive:,}")
print(f"오탐 금액(FP amt 합계)  : ${false_positive_amt:,.0f}")
print(f"탐지율(Recall)          : {recall:.2f}%")

print("\n[원래 자막과 비교] 원래: Recall 96.82% | 탐지 사기 9,344건 | 오탐 3,198건")
print(f"[대본용 자막]\n위험등급 × 거래유형 대응 시뮬레이션\nRecall {recall:.2f}%｜탐지 사기 {detected_fraud:,}건｜오탐 {false_positive:,}건")


전체 실제 사기 건수     : 9,651
탐지된 사기 건수(TP)    : 9,123
오탐 건수(FP)           : 867
오탐 금액(FP amt 합계)  : $414,033
탐지율(Recall)          : 94.53%

[원래 자막과 비교] 원래: Recall 96.82% | 탐지 사기 9,344건 | 오탐 3,198건
[대본용 자막]
위험등급 × 거래유형 대응 시뮬레이션
Recall 94.53%｜탐지 사기 9,123건｜오탐 867건


In [6]:
## 7-2. 같은 '위험' 등급 안에서 유형별 사기율 격차

risk_df = tableau_df[tableau_df["tier_5_final"] == "위험"].copy()

cluster_stats = (
    risk_df.groupby("cluster_type")
    .agg(n=("is_fraud", "size"), fraud_n=("is_fraud", "sum"), fraud_rate=("is_fraud", "mean"))
    .assign(
        fraud_rate_pct=lambda d: d["fraud_rate"] * 100,
        normal_rate_pct=lambda d: 100 - d["fraud_rate"] * 100,
    )
    .sort_values("fraud_rate_pct", ascending=False)
)

print("['위험' 등급 내 유형별 사기율 / 정상거래 비율]")
print(cluster_stats[["n", "fraud_n", "fraud_rate_pct", "normal_rate_pct"]])

high_risk_type = cluster_stats.index[0]
low_risk_type = cluster_stats.index[-1]
high_rate = cluster_stats.loc[high_risk_type, "fraud_rate_pct"]
low_rate = cluster_stats.loc[low_risk_type, "fraud_rate_pct"]
gap = high_rate - low_rate

print(f"\n최고 사기율 유형: '{high_risk_type}'  ({high_rate:.2f}%)")
print(f"최저 사기율 유형: '{low_risk_type}'  ({low_rate:.2f}%)")
print(f"격차              : {gap:.2f}%p")

print("\n[원래 값과 비교] 원래: 58.78% -> 83.56%, 24.78%p 차이")
print("\n[대본용 문장]")
print(
    f"같은 '위험' 등급 안에서도 거래유형에 따라 실제 사기율이 {low_rate:.2f}%에서 {high_rate:.2f}%까지 "
    f"나타나, 유형 간 최대 {gap:.2f}%포인트의 격차가 있음을 확인했습니다."
)
print("\n[대본용 자막]")
print(f"같은 '위험' 등급 내 사기율 격차\n{low_rate:.2f}% -> {high_rate:.2f}%｜{gap:.2f}%p 차이")


['위험' 등급 내 유형별 사기율 / 정상거래 비율]
                   n  fraud_n  fraud_rate_pct  normal_rate_pct
cluster_type                                                  
심야 고액 반복형        192      127         66.1458          33.8542
주간 소액·평소거래형      104       65         62.5000          37.5000
심야 초단기 다회·고속이동형  144       80         55.5556          44.4444
심야 고액·단시간 누적형     92       50         54.3478          45.6522
중간금액·완만 이탈형      300      150         50.0000          50.0000
심야 소액·평소거래형      300      139         46.3333          53.6667
단발성 초고액 이탈형      385      160         41.5584          58.4416

최고 사기율 유형: '심야 고액 반복형'  (66.15%)
최저 사기율 유형: '단발성 초고액 이탈형'  (41.56%)
격차              : 24.59%p

[원래 값과 비교] 원래: 58.78% -> 83.56%, 24.78%p 차이

[대본용 문장]
같은 '위험' 등급 안에서도 거래유형에 따라 실제 사기율이 41.56%에서 66.15%까지 나타나, 유형 간 최대 24.59%포인트의 격차가 있음을 확인했습니다.

[대본용 자막]
같은 '위험' 등급 내 사기율 격차
41.56% -> 66.15%｜24.59%p 차이


In [ ]:
## 7-4. 액션 제안 — 두 유형 좌우 비교 (정상거래 비율)

print(f"고위험 유형(사기율 최고) : '{high_risk_type}'  사기율 {high_rate:.2f}%  정상거래 비율 {100-high_rate:.2f}%")
print(f"저위험 유형(사기율 최저) : '{low_risk_type}'  사기율 {low_rate:.2f}%  정상거래 비율 {100-low_rate:.2f}%")

print("\n[원래 값과 비교] 원래: 심야 고액 정상거래비율 16.44% / 심야 소액 정상거래비율 41.22%")

print("\n[대본용 자막]")
print(f"{high_risk_type}: 정상거래 비율 {100-high_rate:.2f}%\n{low_risk_type}: 정상거래 비율 {100-low_rate:.2f}%")


고위험 유형(사기율 최고) : '심야 고액 반복형'  사기율 66.15%  정상거래 비율 33.85%
저위험 유형(사기율 최저) : '단발성 초고액 이탈형'  사기율 41.56%  정상거래 비율 58.44%

[원래 값과 비교] 원래: 심야 고액 정상거래비율 16.44% / 심야 소액 정상거래비율 41.22%

[대본용 자막]
심야 고액 반복형: 정상거래 비율 33.85%
단발성 초고액 이탈형: 정상거래 비율 58.44%


In [8]:
## 7-5. 정량적 기대효과 — 건수 / 금액

low_risk_normal = risk_df[(risk_df["cluster_type"] == low_risk_type) & (risk_df["is_fraud"] == 0)]
high_risk_fraud = risk_df[(risk_df["cluster_type"] == high_risk_type) & (risk_df["is_fraud"] == 1)]

low_n, low_amt = len(low_risk_normal), low_risk_normal["amt"].sum()
high_n, high_amt = len(high_risk_fraud), high_risk_fraud["amt"].sum()

print(f"'{low_risk_type}' 정상거래  : {low_n:,}건, ${low_amt:,.0f}")
print(f"'{high_risk_type}' 사기거래 : {high_n:,}건, ${high_amt:,.0f}")

print("\n[원래 값과 비교] 원래: 정상거래 183건/$9,531  |  사기거래 249건/$117,143")

print("\n[대본용 문장]")
print(
    f"이를 통해 '{low_risk_type}'의 정상거래 {low_n:,}건, {low_amt:,.0f}달러 규모가 불필요하게 "
    f"장시간 보류되는 고객 불편을 줄일 수 있습니다. 동시에 '{high_risk_type}'에서 확인된 사기거래 "
    f"{high_n:,}건, {high_amt:,.0f}달러 규모를 우선 보류 및 심사 대상으로 관리해 대응 속도와 강도를 "
    f"높일 수 있습니다."
)
print("\n[대본용 자막]")
print(f"불필요한 거래 보류 완화\n정상거래 {low_n:,}건｜${low_amt:,.0f}\n\n고위험 거래 우선 대응\n사기거래 {high_n:,}건｜${high_amt:,.0f}")


'단발성 초고액 이탈형' 정상거래  : 225건, $237,129
'심야 고액 반복형' 사기거래 : 127건, $27,001

[원래 값과 비교] 원래: 정상거래 183건/$9,531  |  사기거래 249건/$117,143

[대본용 문장]
이를 통해 '단발성 초고액 이탈형'의 정상거래 225건, 237,129달러 규모가 불필요하게 장시간 보류되는 고객 불편을 줄일 수 있습니다. 동시에 '심야 고액 반복형'에서 확인된 사기거래 127건, 27,001달러 규모를 우선 보류 및 심사 대상으로 관리해 대응 속도와 강도를 높일 수 있습니다.

[대본용 자막]
불필요한 거래 보류 완화
정상거래 225건｜$237,129

고위험 거래 우선 대응
사기거래 127건｜$27,001


In [10]:
## 요약 — 모든 대본용 문장 한번에 다시 출력

print("=" * 60)
print("[4번] 심야 비율")
print(f"전체 이상거래의 약 {pct_fraud_at_night:.0f}%가 심야에 발생했지만, 심야 거래 자체의 약 {pct_night_is_normal:.0f}%는 정상거래였습니다.")

print("\n[4번] 금액-사기 상관 / 심슨의 역설")
print(f"전체 데이터에서는 거래금액과 이상거래 여부가 약한 양의 상관관계(r={overall_r:.3f})를 보였지만, 업종별로 나누자 {len(reversed_categories)}개 업종에서는 관계의 방향이 반대로 나타났습니다.")

print("\n[7-1] 시뮬레이터 자막")
print(f"Recall {recall:.2f}%｜탐지 사기 {detected_fraud:,}건｜오탐 {false_positive:,}건")

print("\n[7-2] 등급 내 유형별 격차")
print(f"같은 '위험' 등급 내 사기율 격차: {low_rate:.2f}% -> {high_rate:.2f}%｜{gap:.2f}%p 차이")

print("\n[7-4] 두 유형 정상거래 비율")
print(f"{high_risk_type}: {100-high_rate:.2f}%  |  {low_risk_type}: {100-low_rate:.2f}%")

print("\n[7-5] 정량적 기대효과")
print(f"정상거래 {low_n:,}건｜${low_amt:,.0f}  /  사기거래 {high_n:,}건｜${high_amt:,.0f}")
print("=" * 60)


[4번] 심야 비율
전체 이상거래의 약 85%가 심야에 발생했지만, 심야 거래 자체의 약 98%는 정상거래였습니다.

[4번] 금액-사기 상관 / 심슨의 역설
전체 데이터에서는 거래금액과 이상거래 여부가 약한 양의 상관관계(r=0.219)를 보였지만, 업종별로 나누자 6개 업종에서는 관계의 방향이 반대로 나타났습니다.

[7-1] 시뮬레이터 자막
Recall 94.53%｜탐지 사기 9,123건｜오탐 867건

[7-2] 등급 내 유형별 격차
같은 '위험' 등급 내 사기율 격차: 41.56% -> 66.15%｜24.59%p 차이

[7-4] 두 유형 정상거래 비율
심야 고액 반복형: 33.85%  |  단발성 초고액 이탈형: 58.44%

[7-5] 정량적 기대효과
정상거래 225건｜$237,129  /  사기거래 127건｜$27,001
